## Step 04 — Cluster transcripts to detect re-uploaded content

This notebook groups videos whose transcripts contain the same (or very similar) spoken content. A video appearing in a cluster means its transcript closely matches at least one other video in the dataset — a strong signal that the same clip has been re-uploaded across different accounts or posted at different times.

### How the clustering works

Two complementary similarity signals are combined into a single similarity calculation:

1. **Semantic similarity** — each transcript is encoded into a vector using [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), a lightweight sentence-transformer model. This model captures videos that convey the same meaning even if the wording differs slightly.

2. **Containment similarity** — checks whether one transcript is an exact substring of another. This catches cases where a clipped or shortened re-upload is a literal sub-segment of a longer original video.

The two scores are combined with configurable weights (`SEMANTIC_WEIGHT` and `CONTAINMENT_WEIGHT`) and fed into [DBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html). DBSCAN is a clustering algorithm that finds clusters of arbitrary shape and does not require you to specify the number of clusters in advance. Videos that are not similar enough to any cluster are labelled as noise (`cluster_id = -1`) and excluded from the final output.

### Key configuration knobs

| Parameter              | Default | Effect                                                                      |
| ---------------------- | ------- | --------------------------------------------------------------------------- |
| `SEMANTIC_WEIGHT`      | 0.90    | Weight given to semantic similarity in the combined score                   |
| `CONTAINMENT_WEIGHT`   | 0.50    | Weight given to verbatim containment in the combined score                  |
| `SIMILARITY_THRESHOLD` | 0.75    | Minimum combined similarity for two videos to be placed in the same cluster |
| `MIN_CLUSTER_SIZE`     | 2       | Minimum number of videos required to form a cluster                         |

I highly recommend that you play around with these parameters on a small sample of your dataset to see which weights generate your ideal results.

**Input:** `output/transcriptions_translated.csv`  
**Outputs:**

- `output/cluster_summary.csv` — one row per cluster with the cluster ID, size, and an example transcript
- `output/clustered_translations.csv` — full video metadata with cluster assignments (noise excluded)


In [10]:
# INSTALL REQUIREMENTS (one time only):

# (locally)
# %pip install -r ../requirements.txt

# (on google colab)
# !pip install -q sentence-transformers scikit-learn pandas numpy tqdm

import re
import numpy as np
import pandas as pd
import os

from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer

from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity

# (on google colab)
# from google.colab import drive
# drive.mount('/content/drive')


### Clustering Script

Scroll through the section headers inside the cell for a step-by-step breakdown.


In [ ]:
# =============================
# 1. SET CLUSTERING CONFIGS
# =============================

# In google drive: 
# PATH_PREFIX = "/content/drive/MyDrive/tiktok-content-reupload-tracker"

# Locally:
PATH_PREFIX = ".."

FILES_DIRECTORY = f"{PATH_PREFIX}/output"
INPUT_CSV = f"{FILES_DIRECTORY}/transcriptions_translated.csv"

TEXT_COLUMN = "transcript_en"

# Set the weights for the two similarity metrics... 
# Note: Higher = more aggressive grouping
SEMANTIC_WEIGHT = 0.90
CONTAINMENT_WEIGHT = 0.50

SIMILARITY_THRESHOLD = 0.75

MIN_CLUSTER_SIZE = 2

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# ============================================================
# 2. LOAD IN TRANSLATIONS FOR CLUSTERING
# ============================================================

os.listdir(FILES_DIRECTORY)

df = pd.read_csv(INPUT_CSV)

df = df.dropna(subset=[TEXT_COLUMN])

df[TEXT_COLUMN] = (
    df[TEXT_COLUMN]
    .astype(str)
    .str.strip()
)

print(f"\nLoaded {len(df):,} transcripts")


# ============================================================
# 3. CLEAN UP TEXT FOR CLUSTERING
# ============================================================

def normalize(text):

    text = text.lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


df["clean_text"] = df[TEXT_COLUMN].apply(normalize)

texts = df["clean_text"].tolist()


# ============================================================
# 4. GENERATE CLUSTER EMBEDDINGS
# ============================================================

print("\nGenerating embeddings...")

model = SentenceTransformer(MODEL_NAME)

embeddings = model.encode(
    texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Done.")


# ============================================================
# 5. COMPUTE SEMANTIC SIMILARITY
# ============================================================

print("\nComputing semantic similarity matrix...")

semantic_sim = cosine_similarity(embeddings)


# ============================================================
# 6. COMPUTE CONTAINMENT SIMILARITY
# ============================================================

print("Computing containment similarity matrix...")

n = len(texts)

containment_sim = np.zeros((n, n), dtype=np.float32)

for i in tqdm(range(n)):

    t1 = texts[i]

    len1 = len(t1)

    for j in range(i, n):

        t2 = texts[j]

        len2 = len(t2)

        shorter = min(len1, len2)
        longer = max(len1, len2)

        score = 0

        if shorter > 50:

            if t1 in t2 or t2 in t1:

                score = shorter / longer

        containment_sim[i, j] = score
        containment_sim[j, i] = score


# ============================================================
# 7. CALCULATE COMBINED SIMILARITY INDEX
# ============================================================

combined_sim = (
    semantic_sim * SEMANTIC_WEIGHT
    + containment_sim * CONTAINMENT_WEIGHT
)

combined_sim = np.clip(combined_sim, 0, 1) # make sure value doesn't exceed 1

distance_matrix = 1 - combined_sim

distance_matrix = np.clip(distance_matrix, 0, 1) # make sure value doesn't exceed 1


# ============================================================
# 8. CLUSTER BASED ON COMBINED SIMILARITY INDEX 
# ============================================================

print("\nClustering...")

clusterer = DBSCAN(
    eps=(1 - SIMILARITY_THRESHOLD),
    min_samples=MIN_CLUSTER_SIZE,
    metric="precomputed"
)

labels = clusterer.fit_predict(distance_matrix)

df["cluster_id"] = labels


# ============================================================
# 9. BUILD SUMMARY TABLE
# ============================================================

summary_rows = []

for cluster_id in sorted(df["cluster_id"].unique()):

    if cluster_id == -1:
        continue

    cluster_df = df[df["cluster_id"] == cluster_id]

    representative = (
        cluster_df.iloc[0][TEXT_COLUMN][:1000]
    )

    summary_rows.append({
        "cluster_id": cluster_id,
        "size": len(cluster_df),
        "example_text": representative
    })

summary = pd.DataFrame(summary_rows)

# summary = summary.sort_values(
#     "size",
#     ascending=False
# )

# ============================================================
# 10. SAVE RESULTS
# ============================================================

summary.to_csv(
    FILES_DIRECTORY + "/cluster_summary.csv",
    index=False
)

output_clean = df[['file_name', 'cluster_id']]
output_clean['file_name'] = output_clean['file_name'].str.rsplit('.', n=1).str[0] # remove filetype suffix

print(output_clean)

full_translations = pd.read_csv(FILES_DIRECTORY + "/transcriptions_and_translations.csv")
full_translations['meta_id'] = full_translations['meta_id'].astype(str) # convert meta_id to string

clustered_translations = pd.merge(full_translations, output_clean, how='left', left_on='meta_id', right_on='file_name')
clustered_translations = clustered_translations[clustered_translations['cluster_id'] >= 0]

clustered_translations.to_csv(
    FILES_DIRECTORY + "/clustered_translations.csv",
    index=False
)


print("\n================================")
print("CLUSTERING COMPLETE")
print("================================")

print(f"Total transcripts: {len(df):,}")

print(f"Clusters: {len(summary):,}")

print(f"Noise: {(labels == -1).sum():,}")

print("\nLargest Clusters")

display(summary.head())